In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import json
from functools import partial
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

import nomad.io.base as loader
import nomad.stop_detection.utils as utils
from nomad.stop_detection.density_based import seqscan_labels
from nomad.stop_detection.dbscan import ta_dbscan_labels

import nomad.visit_attribution.visit_attribution as visits

from nomad.map_utils import blocks_to_mercator_gdf
from nomad.contact_estimation import compute_stop_detection_metrics
import warnings
warnings.filterwarnings('ignore', message='Input is timezone-naive; assuming UTC')

In [ ]:
with open('config_2_stops.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

In [ ]:
poi_table = gpd.read_parquet(config["buildings_file"]).rename(columns={"id":"location"})
sparse_df = loader.from_file(config["output_files"]["sparse_path"], format="parquet")
sparse_df.drop(columns=['datetime'], inplace=True)

diaries_df = loader.from_file(config["output_files"]["diaries_path"], format="parquet").rename(
    columns={"identifier":"user_id"})

In [ ]:
dist_thresh_values = np.linspace(5, 55, 49)

config["algos"] = {
    **{f"seqscan_dist_thresh_{int(dr)}": {
        "func": seqscan_labels,
        "params": {
            "time_thresh":60,
            "dist_thresh":dr,
            "min_pts":2,
            "back_merge":True,
        },
        "visit_attr":"majority"
    } for dr in dist_thresh_values},
    **{f"st_dbscan_dist_thresh_{int(dr)}": {
        "func": ta_dbscan_labels,
        "params": {
            "time_thresh":60,
            "dist_thresh":dr,
            "min_pts":2,
        },
        "visit_attr":"majority"
    } for dr in dist_thresh_values}    
}

summarize_stops_with_loc = partial(
    utils.summarize_stop,
    x='x',
    y='y',
    timestamp='timestamp',
    keep_col_names=True,
    passthrough_cols=['location'],
    complete_output=False)

## Execution of stop detection

In [ ]:
results_list = []
sparse_df['precomp_locations'] = visits.poi_map(
    sparse_df,
    poi_table=poi_table.loc[poi_table.location.isin(['w-x17-y10', 'r-x19-y11'])],
    data_crs='EPSG:3857',
    max_distance=20, 
    location_id='location',
    x='x',
    y='y'
)

for user in tqdm(diaries_df.user_id.unique(), desc='Processing users'):
    sparse = sparse_df.query("user_id==@user").copy()
    truth = diaries_df.query("user_id==@user").copy()
    
    # Run all st-dbscan parameterizations
    for algo_name, algo_config in config["algos"].items():
        # Run stop detection
        labels = algo_config["func"](sparse, **algo_config["params"], x="x", y="y", timestamp='timestamp')
        sparse['cluster'] = labels
        sparse['location'] = sparse['precomp_locations']
        
        # Map stops to buildings
        sparse["location"] = visits.point_in_polygon(
            sparse, 
            poi_table=poi_table.loc[poi_table.location.isin(['w-x17-y10', 'r-x19-y11'])],
            data_crs='EPSG:3857',
            max_distance=20, 
            location_id='location', 
            method=algo_config["visit_attr"],
            x='x', 
            y='y',
            recompute_location=False
        )

        stops = sparse[sparse.cluster!=-1].groupby('cluster', as_index=False).apply(
            summarize_stops_with_loc, include_groups=False) # no need to deal with empty case
        
        # Compute metrics
        metrics = compute_stop_detection_metrics(
            stops=stops,
            truth=truth,
            user_id=user,
            algorithm=algo_name,
            traj_cols={'location_id': 'location'},
            timestamp='timestamp'
        )
        
        # Add dist_thresh value to metrics
        metrics['dist_thresh'] = algo_config['params']['dist_thresh']
        
        results_list.append(metrics)
        # Clean up temporary columns for next iteration
        sparse.drop(columns=['cluster', 'location'], inplace=True)

# Convert to DataFrame
results_df = pd.DataFrame(results_list)

print(f"Computed metrics for {len(results_df)} user-algorithm combinations")
print(f"Number of unique dist_thresh values: {results_df['dist_thresh'].nunique()}")

## Plotting

In [ ]:
# Plotting function adapted for dist_thresh
def plot_metric_dist_thresh(metric, title='', save_path='figures'):
    """Plot a metric vs dist_thresh for Lachesis."""
    chart_df = results_df.groupby(['dist_thresh'])[metric].agg(['mean', 'std']).reset_index()

    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Add scatter plot background (individual data points)
    sns.scatterplot(data=results_df, x='dist_thresh', y=metric,
                   alpha=0.1, s=45, ax=ax, color='steelblue')
    
    # Add line plot (mean values)
    ax.plot(chart_df['dist_thresh'], chart_df['mean'],
            color='darkblue', linewidth=3, label='Mean')
    
    #Add confidence band (mean ± std)
    ax.fill_between(chart_df['dist_thresh'],
                    chart_df['mean'] - chart_df['std'],
                    chart_df['mean'] + chart_df['std'],
                    alpha=0.2, color='steelblue')
    
    # Styling
    ax.set_xlabel('Distance Threshold (meters)', fontsize=15, labelpad=10)
    ax.set_ylabel(title, fontsize=15, labelpad=10)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=16, pad=15, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.6, linewidth=0.7)
    ax.tick_params(axis='both', which='major', labelsize=13, length=6, width=1.2)
    ax.minorticks_on()
    ax.legend(fontsize=12, frameon=True, loc='best')
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/dbscan_dist_thresh_{metric}.svg", bbox_inches='tight')
    plt.savefig(f"{save_path}/dbscan_dist_thresh_{metric}.png", dpi=600, bbox_inches='tight')
    plt.show()
    plt.close()

## Seqscan

In [ ]:
plot_metric_dist_thresh("recall", "Accuracy")

In [ ]:
chart_df = results_df.groupby(['dist_thresh'])['recall'].agg(['mean', 'std']).reset_index()
chart_df['mean'].max()

In [ ]:
results_df.query("14 > dist_thresh > 12 and recall < 0.4")